In [19]:
import pandas as pd
import altair as alt
alt.renderers.enable('html')
import numpy as np
import plotly.graph_objects as go
from plotly.colors import sample_colorscale
import nbformat


Reading Data

In [20]:
df = pd.read_csv('analysis_dataset.csv')
df.head()

,p_recall,timestamp,delta,user_id,learning_language,ui_language,lexeme_id,lexeme_string,history_seen,history_correct,session_seen,session_correct
0,0.0,1362434189,342767,u:_6K,en,es,b974624f112084b39586380fe4aaf32a,man/man<n><sg>,16,14,1,0
1,1.0,1362698937,106662,u:i2TA,en,es,928787744a962cd4ec55c1b22cedc913,eats/eat<vblex><pri><p3><sg>,5,3,3,3
2,1.0,1362210918,51859,u:iIYs,es,en,22e3fc5f089cf47dae927dfddcce2844,ellos/prpers<prn><tn><p3><m><pl>,8,8,1,1
3,1.0,1362347852,3610438,u:gQ71,es,en,f9d3186a3a829083942aa09a75d1aae6,leen/leer<vblex><pri><p3><pl>,2,2,1,1
4,0.5,1362870047,581551,u:iLj2,en,es,c76d36f47e394429d9c14d2088e4a85c,phase/phase<n><sg>,3,3,2,1


In [21]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   p_recall           5000 non-null   float64
 1   timestamp          5000 non-null   int64  
 2   delta              5000 non-null   int64  
 3   user_id            5000 non-null   str    
 4   learning_language  5000 non-null   str    
 5   ui_language        5000 non-null   str    
 6   lexeme_id          5000 non-null   str    
 7   lexeme_string      5000 non-null   str    
 8   history_seen       5000 non-null   int64  
 9   history_correct    5000 non-null   int64  
 10  session_seen       5000 non-null   int64  
 11  session_correct    5000 non-null   int64  
dtypes: float64(1), int64(6), str(5)
memory usage: 468.9 KB


In [22]:
df.describe()

,p_recall,timestamp,delta,history_seen,history_correct,session_seen,session_correct
count,5000.000000,5.000000e+03,5.000000e+03,5000.00000,5000.00000,5000.000000,5000.000000
mean,0.890519,1.362586e+09,7.678062e+05,24.47760,21.70040,1.777400,1.599600
std,0.280194,2.914586e+05,2.408921e+06,143.04737,126.89792,1.280069,1.248836
min,0.000000,1.362083e+09,2.000000e+00,1.00000,1.00000,1.000000,0.000000
25%,1.000000,1.362340e+09,5.350000e+02,3.00000,3.00000,1.000000,1.000000
50%,1.000000,1.362588e+09,7.882200e+04,6.00000,6.00000,1.000000,1.000000
75%,1.000000,1.362842e+09,4.747660e+05,15.00000,13.00000,2.000000,2.000000
max,1.000000,1.363105e+09,3.170928e+07,5657.00000,5370.00000,16.000000,16.000000


In [23]:
df['history_accuracy'] = df['history_correct'] / df['history_seen'] # Creating new metrics/column
df['datetime'] = pd.to_datetime(df['timestamp'], unit='s')
df['day_of_week'] = df['datetime'].dt.day_name()
df['day_type'] = np.where(df['datetime'].dt.dayofweek >= 5, 'Weekend', 'Weekday')

lang_names = {'de': 'German', 'en': 'English', 'es': 'Spanish',
              'fr': 'French', 'it': 'Italian', 'pt': 'Portuguese'}
df.head()

/var/folders/8r/29f8vpsx10b54lc3kd02d4500000gn/T/ipykernel_79258/1305174224.py:4: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  df['day_type'] = np.where(df['datetime'].dt.dayofweek >= 5, 'Weekend', 'Weekday')


,p_recall,timestamp,delta,user_id,learning_language,ui_language,lexeme_id,lexeme_string,history_seen,history_correct,session_seen,session_correct,history_accuracy,datetime,day_of_week,day_type
0,0.0,1362434189,342767,u:_6K,en,es,b974624f112084b39586380fe4aaf32a,man/man<n><sg>,16,14,1,0,0.875,2013-03-04 21:56:29,Monday,Weekday
1,1.0,1362698937,106662,u:i2TA,en,es,928787744a962cd4ec55c1b22cedc913,eats/eat<vblex><pri><p3><sg>,5,3,3,3,0.600,2013-03-07 23:28:57,Thursday,Weekday
2,1.0,1362210918,51859,u:iIYs,es,en,22e3fc5f089cf47dae927dfddcce2844,ellos/prpers<prn><tn><p3><m><pl>,8,8,1,1,1.000,2013-03-02 07:55:18,Saturday,Weekend
3,1.0,1362347852,3610438,u:gQ71,es,en,f9d3186a3a829083942aa09a75d1aae6,leen/leer<vblex><pri><p3><pl>,2,2,1,1,1.000,2013-03-03 21:57:32,Sunday,Weekend
4,0.5,1362870047,581551,u:iLj2,en,es,c76d36f47e394429d9c14d2088e4a85c,phase/phase<n><sg>,3,3,2,1,1.000,2013-03-09 23:00:47,Saturday,Weekend


Replacing Abbreviations

In [24]:
df['learning_language'].unique().tolist()

['en', 'es', 'de', 'fr', 'it', 'pt']

In [25]:
conversions = {'en':'English', 'es':'Spanish', 'de': 'German', 'fr':'French', 'it':'Italian', 'pt':'Portuguese'}
df_renamed = df.copy()
df_renamed['learning_language'] = df_renamed['learning_language'].replace(conversions)
df_renamed


,p_recall,timestamp,delta,user_id,learning_language,ui_language,lexeme_id,lexeme_string,history_seen,history_correct,session_seen,session_correct,history_accuracy,datetime,day_of_week,day_type
0,0.0,1362434189,342767,u:_6K,English,es,b974624f112084b39586380fe4aaf32a,man/man<n><sg>,16,14,1,0,0.875000,2013-03-04 21:56:29,Monday,Weekday
1,1.0,1362698937,106662,u:i2TA,English,es,928787744a962cd4ec55c1b22cedc913,eats/eat<vblex><pri><p3><sg>,5,3,3,3,0.600000,2013-03-07 23:28:57,Thursday,Weekday
2,1.0,1362210918,51859,u:iIYs,Spanish,en,22e3fc5f089cf47dae927dfddcce2844,ellos/prpers<prn><tn><p3><m><pl>,8,8,1,1,1.000000,2013-03-02 07:55:18,Saturday,Weekend
3,1.0,1362347852,3610438,u:gQ71,Spanish,en,f9d3186a3a829083942aa09a75d1aae6,leen/leer<vblex><pri><p3><pl>,2,2,1,1,1.000000,2013-03-03 21:57:32,Sunday,Weekend
4,0.5,1362870047,581551,u:iLj2,English,es,c76d36f47e394429d9c14d2088e4a85c,phase/phase<n><sg>,3,3,2,1,1.000000,2013-03-09 23:00:47,Saturday,Weekend
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4995,1.0,1362477432,44471,u:iW-8,Italian,en,e525865ed08dc764757a8c767412a33c,arrivederci/arrivederci<ij>,4,3,5,5,0.750000,2013-03-05 09:57:12,Tuesday,Weekday
4996,1.0,1362673149,494,u:iD3z,English,es,142ad0f32d08728de0f9955df5b83f32,an/a<det><ind><sg>,5,4,1,1,0.800000,2013-03-07 16:19:09,Thursday,Weekday
4997,1.0,1362844218,188220,u:iT8o,Spanish,en,9a923eb44d7eab8152760002d9100c7a,sus/suyo<det><pos><mf><pl>,7,6,1,1,0.857143,2013-03-09 15:50:18,Saturday,Weekend
4998,1.0,1363013171,124,u:eDQc,Italian,en,7e6d656a6c38aca23d98acc796a35720,beve/bere<vblex><pri><p3><sg>,8,6,2,2,0.750000,2013-03-11 14:46:11,Monday,Weekday


Analyzing differences in historical learning performance across learning languages based on UI language
* NOTES
- Average accuracy vs language (Richy)
- Change labels to full languages
- Sort ranking
- Remove color


In [ ]:
chart = alt.Chart(df_renamed).mark_bar().encode(
    x=alt.X('learning_language:N', title='Learning Language', sort='-y'),
    y=alt.Y('mean(history_accuracy):Q',
            scale=alt.Scale(domain=[0.85, 0.95], zero=False),
            axis=alt.Axis(format='.0%'),
            title='Mean Historical Accuracy'),
    tooltip=alt.Tooltip('mean(history_accuracy):Q', format='.2f')
).properties(
    title='Mean Historical Accuracy VS Language being Learned',
    width=300,
    height=300
)

chart.save('charts/chart1.html')
chart

As learners encounter the same vocabulary item more times historically, does their current-session recall improve?

In [ ]:
# History Bins
bins = df.copy()

bins['history_group'] = pd.cut(
    bins['history_seen'],
    bins=[0, 4, 9, 19, 49, float("inf")],
    labels=['1–4', '5–9', '10–19', '20–49', '50+']
)


In [ ]:
# Add full language names using the map
bins['language'] = bins['learning_language'].map(lang_names)

languages = sorted(bins['language'].dropna().unique())
bins['history_order'] = bins['history_group'].cat.codes

lang_dropdown = alt.binding_select(options=[None] + languages,
                                   labels=['All'] + languages,
                                   name='Learning Language: ')
lang_select = alt.selection_point(fields=['language'], bind=lang_dropdown)

line = alt.Chart(bins).mark_line().encode(
    x=alt.X('history_group:O',
            sort=['1–4', '5–9', '10–19', '20–49', '50+'],
            title='History Seen'),
    y=alt.Y('mean(p_recall):Q',
            scale=alt.Scale(zero=False, nice=True, padding=20),
            axis=alt.Axis(format='.1%'),
            title='Average Recall'),
    order=alt.Order('min(history_order):Q')
).add_params(
    lang_select
).transform_filter(
    lang_select
).properties(
    title='Average Recall by History Seen',
    width=500,
    height=300
)

point = alt.Chart(bins).mark_point().encode(

    x=alt.X('history_group:O',
            sort=['1–4', '5–9', '10–19', '20–49', '50+'],
            title='History Seen'),
    y=alt.Y('mean(p_recall):Q',
            scale=alt.Scale(zero=False, nice=True, padding=20),
            axis=alt.Axis(format='.1%'),
            title='Average Recall')
).transform_filter(
    lang_select
).properties(
    title='Average Recall by History Seen',
    width=500,
    height=300
)

overlay = point + line
overlay.save('charts/chart2.html')
overlay

Does repeated exposure to a vocabulary item within the current session correspond with higher recall?


In [ ]:
# Session Bins
bins = df.copy()

bins['session_group'] = pd.cut(
    bins['session_seen'],
    bins=[0, 1, 2, 3, 4, float('inf')],
    labels=['1', '2', '3', '4', '5+']
)

In [ ]:
line = alt.Chart(bins).mark_line().encode(
    x=alt.X('session_group:O',
            sort = ['1', '2', '3', '4', '5+'],
            title='Session Seen'),
    y=alt.Y('mean(p_recall):Q',
            scale=alt.Scale(domain=[0.85, 0.95], zero=False),
            axis=alt.Axis(format='.0%'),
            title='Average Recall'),
).properties(
    title='Average Recall vs Session Seen',
    width=500,
    height=300
)

point = alt.Chart(bins).mark_point().encode(
    x=alt.X('session_group:O',
            sort = ['1', '2', '3', '4', '5+'],
            title='Session Seen'),
    y=alt.Y('mean(p_recall):Q',
            scale=alt.Scale(domain=[0.85, 0.95], zero=False),
            axis=alt.Axis(format='.0%'),
            title='Average Recall'),
).properties(
    title='Average Recall vs Session Seen',
    width=500,
    height=300
)

combined = point + line
combined.display()

In [ ]:
dow_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday',
             'Friday', 'Saturday', 'Sunday']

dow = (df.groupby(['learning_language', 'day_of_week'])
         .agg(mean_recall=('p_recall', 'mean'),
              n_obs=('p_recall', 'size'))
         .reset_index())
dow['language'] = dow['learning_language'].map(lang_names)

# Overall average per day, across all observations
dow_overall = (df.groupby('day_of_week')
                 .agg(mean_recall=('p_recall', 'mean'),
                      n_obs=('p_recall', 'size'))
                 .reset_index())

dow_languages = sorted(dow['language'].dropna().unique())

dow_select = alt.selection_point(fields=['language'], bind='legend')

x_enc = alt.X('day_of_week:O', sort=dow_order, title='Day of Week',
              axis=alt.Axis(labelAngle=0))
y_enc = alt.Y('mean_recall:Q', scale=alt.Scale(zero=False, padding=20),
              axis=alt.Axis(format='.1%'), title='Mean Recall')

# Per-language lines (clickable)
lang_lines = alt.Chart(dow).mark_line(point=True, strokeWidth=2.5).encode(
    x=x_enc,
    y=y_enc,
    color=alt.Color('language:N', title='Learning Language',
                    scale=alt.Scale(domain=dow_languages)),
    opacity=alt.condition(dow_select, alt.value(1), alt.value(0.15)),
    tooltip=['language:N', 'day_of_week:N',
             alt.Tooltip('mean_recall:Q', format='.2%'),
             alt.Tooltip('n_obs:Q', format=',')]
).add_params(
    dow_select
)

# Overall average overlay, with its own legend entry
overall_line = alt.Chart(dow_overall).mark_line(
    color='black', strokeWidth=2,
    point=alt.OverlayMarkDef(color='black', filled=True, size=40)
).encode(
    x=x_enc,
    y=y_enc,
    strokeDash=alt.StrokeDash(
        'series:N',
        scale=alt.Scale(domain=['All languages'], range=[[6, 4]]),
        legend=alt.Legend(title=None, symbolType='circle')
    ),
    tooltip=[alt.Tooltip('day_of_week:N', title='Day'),
             alt.Tooltip('mean_recall:Q', format='.2%', title='Overall Mean Recall'),
             alt.Tooltip('n_obs:Q', format=',', title='Observations')]
).transform_calculate(
    series="'All languages'"
)

dow_chart = (lang_lines + overall_line).properties(
    title='Mean Recall by Day of Week',
    width=500, height=300
)

dow_chart.save('charts/chart5.html')
dow_chart

In [ ]:
flows = (df.groupby(['ui_language', 'learning_language'])
           .agg(n_users=('user_id', 'nunique'),
                n_obs=('p_recall', 'size'),
                mean_recall=('p_recall', 'mean'))
           .reset_index())

ui_langs = sorted(flows['ui_language'].unique())
learn_langs = sorted(flows['learning_language'].unique())

node_labels = ([f"{lang_names[l]} (UI)" for l in ui_langs] +
               [f"{lang_names[l]} (learning)" for l in learn_langs])

ui_index = {l: i for i, l in enumerate(ui_langs)}
learn_index = {l: i + len(ui_langs) for i, l in enumerate(learn_langs)}

# Color each link by mean recall
r_min, r_max = flows['mean_recall'].min(), flows['mean_recall'].max()
scaled = (flows['mean_recall'] - r_min) / (r_max - r_min) if r_max > r_min else 0.5
link_colors = [c.replace('rgb', 'rgba').replace(')', ', 0.6)')
               for c in sample_colorscale('Blues', list(0.3 + 0.7 * scaled))]

fig = go.Figure(go.Sankey(
    arrangement='snap',
    node=dict(
        label=node_labels,
        pad=20,
        thickness=18,
        color='#9aa5b1',
        line=dict(color='white', width=0.5)
    ),
    link=dict(
        source=flows['ui_language'].map(ui_index),
        target=flows['learning_language'].map(learn_index),
        value=flows['n_users'],
        color=link_colors,
        customdata=flows[['mean_recall', 'n_obs']].values,
        hovertemplate=('%{source.label} → %{target.label}<br>'
                       'Users: %{value:,}<br>'
                       'Observations: %{customdata[1]:,}<br>'
                       'Mean recall: %{customdata[0]:.1%}<extra></extra>')
    )
))

fig.update_layout(
    title='Learners by UI Language → Learning Language<br>'
          '<sup>Link width = number of users; darker links = higher mean recall</sup>',
    font_size=12,
    width=800,
    height=500
)

fig.write_html('charts/chart6.html')

In [ ]:
fig